In [9]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.messages import HumanMessage
from langchain_salesforce import SalesforceTool

load_dotenv()

username = os.getenv("SALESFORCE_USERNAME")
password = os.getenv("SALESFORCE_PASSWORD", "your-password")
security_token = os.getenv("SALESFORCE_SECURITY_TOKEN", "your-security-token")
domain = os.getenv("SALESFORCE_DOMAIN", "login")

# Initialize the Salesforce tool
tool = SalesforceTool(
    username=username, password=password, security_token=security_token, domain=domain
)


In [10]:
llm = ChatGroq(
    model="llama-3.1-8b-instant", #llama-3.1-8b-instant,openai/gpt-oss-120b,openai/gpt-oss-20b
    temperature=0.0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# First, let's query some contacts to get real data
contacts_query = {
    "operation": "query",
    "query": "SELECT Id, Name, Email, Phone FROM Contact LIMIT 3",
}

contacts_result = tool.invoke(contacts_query)
print(contacts_result)

OrderedDict({'totalSize': 3, 'done': True, 'records': [OrderedDict({'attributes': OrderedDict({'type': 'Contact', 'url': '/services/data/v59.0/sobjects/Contact/003dM00001uzLjVQAU'}), 'Id': '003dM00001uzLjVQAU', 'Name': 'Rose Gonzalez', 'Email': 'rose@edge.com', 'Phone': '(512) 757-6000'}), OrderedDict({'attributes': OrderedDict({'type': 'Contact', 'url': '/services/data/v59.0/sobjects/Contact/003dM00001uzLjWQAU'}), 'Id': '003dM00001uzLjWQAU', 'Name': 'Sean Forbes', 'Email': 'sean@edge.com', 'Phone': '(512) 757-6000'}), OrderedDict({'attributes': OrderedDict({'type': 'Contact', 'url': '/services/data/v59.0/sobjects/Contact/003dM00001uzLjXQAU'}), 'Id': '003dM00001uzLjXQAU', 'Name': 'Jack Rogers', 'Email': 'jrogers@burlington.com', 'Phone': '(336) 222-7000'})]})


In [11]:
# Now let's use the LLM to analyze and summarize the contact data
if contacts_result and "records" in contacts_result:
    contact_data = contacts_result["records"]

    # Create a message asking the LLM to analyze the contact data
    analysis_prompt = f"""
    Please analyze the following Salesforce contact data and provide insights:

    Contact Data: {contact_data}

    Please provide:
    1. A summary of the contacts
    2. Any patterns you notice
    3. Suggestions for data quality improvements
    """

    message = HumanMessage(content=analysis_prompt)
    analysis_result = llm.invoke([message])

    print("\nLLM Analysis:")
    print(analysis_result.content)


LLM Analysis:
**Summary of the Contacts**

Based on the provided Salesforce contact data, there are three contacts:

1. **Rose Gonzalez**: Email - `rose@edge.com`, Phone - `(512) 757-6000`
2. **Sean Forbes**: Email - `sean@edge.com`, Phone - `(512) 757-6000`
3. **Jack Rogers**: Email - `jrogers@burlington.com`, Phone - `(336) 222-7000`

**Patterns Noticed**

1. **Shared Phone Number**: Both Rose Gonzalez and Sean Forbes share the same phone number, `(512) 757-6000`. This could indicate a data entry error or a deliberate attempt to associate them with the same phone number.
2. **Different Email Domains**: Rose Gonzalez and Sean Forbes have email addresses from the same domain (`edge.com`), while Jack Rogers has an email address from a different domain (`burlington.com`). This could indicate different affiliations or roles.
3. **Similar Contact IDs**: The contact IDs for Rose Gonzalez and Sean Forbes have a similar pattern (`003dM00001uzLjVQAU` and `003dM00001uzLjWQAU`), which could sug